In [1]:
%load_ext autoreload
%autoreload 2

In [19]:
import pandas as pd
import json
from pathlib import Path

# --- PATH CONFIGURATION ---
GENERATIONS_FOLDER = Path.cwd() / "generations"
GENERATIONS_REGIONS = ['output_montana', 'output_wyoming', 'output_new_mexico']
INPUT_DIRS = {region: GENERATIONS_FOLDER / region / "ra_diagnosed/csv" for region in GENERATIONS_REGIONS}
MEDS_FILE_NAME = "medications.csv"
MEDS_FILE_PATHS = {region: input_dir / MEDS_FILE_NAME for region, input_dir in INPUT_DIRS.items()}

# --- UHC POLICY CLINICAL MAPPINGS (Focus: Section A - Adalimumab) ---
DMARDS = ['methotrexate', 'leflunomide', 'sulfasalazine', 'hydroxychloroquine']
TARGET_BIOLOGIC = ['adalimumab', 'amjevita', 'hyrimoz']

In [21]:
def check_eligibility(group):
    """
    Evaluates patient history against UHC Section A criteria for Adalimumab.
    Handles Timezone-naive conversion to avoid subtraction errors.
    """
    group = group.sort_values('START')
    
    failed_dmards = []
    
    now = pd.Timestamp.now().tz_localize(None)
    
    for _, row in group.iterrows():
        med_desc = row['DESCRIPTION'].lower()
        
        start = pd.to_datetime(row['START']).tz_localize(None)
        
        if pd.notna(row['STOP']) and row['STOP'] != '':
            stop = pd.to_datetime(row['STOP']).tz_localize(None)
            is_stopped = True
        else:
            stop = now
            is_stopped = False
        
        duration_days = (stop - start).days
        
        # --- POLICY RULE: DMARD TRIAL (Section A.1.a.2.a) ---
        # Rule: Minimum 3-month trial (90 days)
        if any(d in med_desc for d in DMARDS):
            if duration_days >= 90:
                failed_dmards.append({
                    "med": med_desc,
                    "duration": duration_days,
                    "status": "Completed Trial" if is_stopped else "Currently Active"
                })

    # --- CLASSIFICATION ---
    if len(failed_dmards) >= 1:
        return "READY_FOR_ADALIMUMAB_PA"

    return "CONTINUE_DMARD_TRIAL"

def main():
    for region, med_file_path in MEDS_FILE_PATHS.items():
        if not med_file_path.exists():
            print(f"File not found: {med_file_path}")
            continue
            
        print(f"\n--- UHC COMPLIANCE AUDIT (ADALIMUMAB): {region.upper()} ---")
        df = pd.read_csv(med_file_path)
        
        relevant_pattern = '|'.join(DMARDS + TARGET_BIOLOGIC)
        df_ra_meds = df[df['DESCRIPTION'].str.contains(relevant_pattern, case=False, na=False)]
        
        results = {
            "READY_FOR_ADALIMUMAB_PA": [], 
            "CONTINUE_DMARD_TRIAL": []
        }
        
        for patient_id, group in df_ra_meds.groupby('PATIENT'):
            status = check_eligibility(group)
            results[status].append(patient_id)

        print(f"{'UHC RULE STATUS':<30} | {'COUNT':<6} | {'NEXT CLINICAL STEP'}")
        print("-" * 85)
        print(f"{'Eligible for Step-Up (PA)':<30} | {len(results['READY_FOR_ADALIMUMAB_PA']):<6} | Extract evidence & Generate PA")
        print(f"{'Under 90-day DMARD trial':<30} | {len(results['CONTINUE_DMARD_TRIAL']):<6} | Clinical monitoring only")
        
        output_file = f"adalimumab_candidates_{region}.json"
        with open(output_file, "w") as f:
            json.dump(results, f, indent=2)
        print(f"\n[FILE] Saved to {output_file}")

if __name__ == "__main__":
    main()


--- UHC COMPLIANCE AUDIT (ADALIMUMAB): OUTPUT_MONTANA ---
UHC RULE STATUS                | COUNT  | NEXT CLINICAL STEP
-------------------------------------------------------------------------------------
Eligible for Step-Up (PA)      | 139    | Extract evidence & Generate PA
Under 90-day DMARD trial       | 2      | Clinical monitoring only

[FILE] Saved to adalimumab_candidates_output_montana.json

--- UHC COMPLIANCE AUDIT (ADALIMUMAB): OUTPUT_WYOMING ---
UHC RULE STATUS                | COUNT  | NEXT CLINICAL STEP
-------------------------------------------------------------------------------------
Eligible for Step-Up (PA)      | 131    | Extract evidence & Generate PA
Under 90-day DMARD trial       | 2      | Clinical monitoring only

[FILE] Saved to adalimumab_candidates_output_wyoming.json

--- UHC COMPLIANCE AUDIT (ADALIMUMAB): OUTPUT_NEW_MEXICO ---
UHC RULE STATUS                | COUNT  | NEXT CLINICAL STEP
-------------------------------------------------------------------